In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scipy.io as sio
from dataclasses import dataclass
from typing import List, Tuple
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
from stl import mesh
import matplotlib.pyplot as plt
import re

In [2]:
import sys
import os

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(fr'H:\codes\tidy3d'))

from AutomationModule import * 

import AutomationModule as AM

In [3]:
tidy3dAPI = os.environ["API_TIDY3D_KEY"]


In [4]:
lambda_ranges =[np.array([8,2.5])]

In [5]:
n_eff=1
run = True

In [6]:
folder_path = rf"../Structures"
project_name = "202606011_Beam_Spreading_12x_L_1_freqs_1700"
postprocess_results = []
runtimes_ps = [20e-12]*3
min_steps_per_lambda = 15
cuts=[1]
h5_bg = None
ref = True
for l,lambdas in enumerate(lambda_ranges): 
    for dirpath, dirnames, filenames in os.walk(folder_path):
        try:
            for filename in filenames:
                if filename.endswith(".h5") and filename=="n_2.90_ff_0.2237.h5":
                    ff = float(re.search(r'ff_([+-]?\d+(?:\.\d+)?)', filename).group(1))
                    n_value = float(re.search(r'n_([+-]?\d+(?:\.\d+)?)', filename).group(1))
                    for cut in cuts:
                        if not (Path(filename).suffix==".h5" or Path(filename).suffix==".stl"):
                            continue 
                        if os.path.isfile(os.path.join(dirpath, filename)):
                            file=os.path.join(dirpath, filename)
                            structure_1 = AM.loadAndRunStructure(key = tidy3dAPI, file_path=file
                                                            ,direction="z", lambda_range=lambdas,
                                                            box_size=14.3,runtime_ps=runtimes_ps[l],min_steps_per_lambda=min_steps_per_lambda,
                                                           scaling=1,shuoff_condtion=1e-20, verbose=True, multiplication_factor=12, multiplicate_size=True,
                                                           monitors=["time_monitor"], flux_monitor_position=18,cell_size_manual=45,
                                                           freqs=1700, width=0.4,sim_bg=n_eff**2,
                                                           cut_condition=cut, source="planewave", absorbers=130, use_permittivity=False,sim_name=rf"{Path(filename).stem}_size_{cut}" + (rf"_bg_{h5_bg}" if h5_bg else ""),h5_bg=h5_bg,
                                                           )
                            
                            
                            f_min, f_max = td.C_0/lambdas[0], td.C_0/lambdas[1]
                            f0 = (f_min + f_max) / 2
                            source_def = td.PlaneWave(
                                source_time = td.GaussianPulse(
                                    freq0=f0,
                                    fwidth=0.5*(f_max - f_min),
                                    offset=10,
                                    phase=0
                                ),
                                size= (2.5, 
                                      2.5, 
                                      0) 
                                      ,
                                center=( 0, 
                                         0, 
                                        (-structure_1.t_slab_z/2 -1)),
                                direction='+',
                                
                                pol_angle=structure_1.pol_angle,
                                name='planewave',
                                )
                            
                            monitorField_exit = td.FieldMonitor(
                                    center=[0,0,structure_1.t_slab_z/2],
                                    size=[
                                            structure_1.t_slab_x,
                                            structure_1.t_slab_y,
                                            0
                                        ],
                                    interval_space = (3,3,3),

                                    fields=["Ex", "Ey", "Ez"],
                                    freqs=structure_1.monitor_freqs,
                                    name="monitorField_exit",
                                )
                            
                            sim = structure_1.sim 
                            sim=sim.copy(update={"sources":[source_def],"monitors":[monitorField_exit]})

                            if run:
                                folder_desc = rf"../../../../data/{project_name}/n_{n_value:.2f}"
                                os.makedirs(folder_desc, exist_ok=True)
                                sim_name=rf"LSU_{Path(filename).stem}_size_{cut}_lambda_{lambdas[1]}_{lambdas[0]}_sim_bg_{n_eff**2:.2f}"
                                if os.path.exists(os.path.join(folder_desc, sim_name+".txt")):
                                    print("Exist!")
                                else:
                                    id =web.upload(sim, folder_name=project_name,task_name=sim_name, verbose=True)
                                    ids = '\n' + id
                                    with open(os.path.join(folder_desc, sim_name+".txt"), "w") as file:
                                        # Write the string to the file
                                        file.write(ids)
                                    web.start(task_id = id)
                                    web.monitor(id)
                            else: 
                                id =web.upload(sim, verbose=True) 
                                web.estimate_cost(id)  

        except Exception as e:
            print(f"Error processing {dirpath}: {e}")

        


Configured successfully.


15:47:02 W. Europe Daylight Time WARNING: Structure at 'structures[0]' has      
                                 bounds that extend exactly to simulation edges.
                                 This can cause unexpected behavior. If         
                                 intending to extend the structure to infinity  
                                 along one dimension, use td.inf as a size      
                                 variable instead to make this explicit.        

                                 WARNING: Suppressed 47 WARNING messages.       

16:33:53 W. Europe Daylight Time WARNING: Structure at 'structures[0]' has      
                                 bounds that extend exactly to simulation edges.
                                 This can cause unexpected behavior. If         
                                 intending to extend the structure to infinity  
                                 along one dimension, use td.inf as a size      
                                 variable instead to make this explicit.        

                                 WARNING: Suppressed 47 WARNING messages.       

...

Output()

Output()

21:26:48 W. Europe Daylight Time status = postprocess

22:53:46 W. Europe Daylight Time status = success

22:53:48 W. Europe Daylight Time View simulation result at                      
                                 ]8;id=777726;https://tidy3d.simulation.cloud/workbench?taskId=fdve-63b48d34-87e7-414d-9d98-048e8a0439b6\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=506473;https://tidy3d.simulation.cloud/workbench?taskId=fdve-63b48d34-87e7-414d-9d98-048e8a0439b6\task]8;;\
                                 ]8;id=506473;https://tidy3d.simulation.cloud/workbench?taskId=fdve-63b48d34-87e7-414d-9d98-048e8a0439b6\Id]8;;\]8;id=777726;https://tidy3d.simulation.cloud/workbench?taskId=fdve-63b48d34-87e7-414d-9d98-048e8a0439b6\=]8;;\]8;id=400516;https://tidy3d.simulation.cloud/workbench?taskId=fdve-63b48d34-87e7-414d-9d98-048e8a0439b6\fdve]8;;\]8;id=777726;https://tidy3d.simulation.cloud/workbench?taskId=fdve-63b48d34-87e7-414d-9d98-048e8a0439b6\-63b48d34-87e7-414d-9d98-048e8a0439b6']8;;\.